### Phase 9: XGBoost rank:pairwise, a Direct AUC Optimization




#### Motivation

All previous models used binary:logistic objective. This minimizes log-loss, not AUC. The model learns to predict calibrated probabilities, and we hope good probabilities  good AUC ranking.

rank:pairwise directly optimizes the pairwise ranking loss. It learns to rank positive examples above negative examples, which is exactly what AUC measures. This is a fundamentally different and more direct optimization objective for our competition metric.

Why we haven't tried this until now:  
It requires special handling. rank:pairwise outputs raw scores (not probabilities), needs group/query structure, and can't use scale_pos_weight. We address all of these below.


## Key Differences from Previous Models

| | binary:logistic (previous) | rank:pairwise (this notebook) |
|---|---|---|
| Objective | Minimize log-loss | Maximize pairwise AUC |
| Output | Calibrated probabilities [0,1] | Raw ranking scores |
| Class imbalance | scale_pos_weight=33.4 | Not applicable — use subsampling |
| Evaluation metric | AUC (indirect) | AUC (direct) |

## 1. Imports & Setup

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import xgboost as xgb
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# ─ Paths 
BASE_DIR   = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR   = os.path.join(BASE_DIR, 'data')
OUT_DIR    = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, 'train_features.csv')
TEST_PATH  = os.path.join(DATA_DIR, 'test_features.csv')
SUB_PATH   = os.path.join(OUT_DIR,  'submission_rankpairwise.csv')

SEEDS          = [42, 123, 456, 789, 2026]
N_FOLDS        = 5
EARLY_STOPPING = 50

print(f'XGBoost version: {xgb.__version__}')

XGBoost version: 2.0.3


## 2. Load Data

In [2]:
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

FEATURE_COLS = [c for c in train.columns if c not in ['id', 'order_placed']]

X      = train[FEATURE_COLS].values
y      = train['order_placed'].values
X_test = test[FEATURE_COLS].values

print(f'Train : {X.shape} | Positives: {y.sum():,} ({y.mean()*100:.2f}%)')
print(f'Test  : {X_test.shape}')
print(f'Features: {len(FEATURE_COLS)}')

Train : (297236, 29) | Positives: 8,644 (2.91%)
Test  : (99639, 29)
Features: 29


## 3. rank:pairwise XGBoost


In [3]:
# rank:pairwise params — adapted from our best binary:logistic params
# Key changes: objective, eval_metric, remove scale_pos_weight
RANK_PARAMS = {
    'objective'        : 'rank:pairwise',
    'eval_metric'      : 'auc',
    'n_estimators'     : 1500,
    'learning_rate'    : 0.01381105528962201,
    'max_depth'        : 9,
    'min_child_weight' : 46,
    'subsample'        : 0.9280864631542471,
    'colsample_bytree' : 0.68577731284726,
    'reg_alpha'        : 2.75787843275108,
    'reg_lambda'       : 3.98468850175631,
    'n_jobs'           : -1,
    'verbosity'        : 0,
}

def run_cv_rank(seed):
    params = {**RANK_PARAMS, 'random_state': seed}
    skf        = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    oof_preds  = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    fold_aucs  = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        # rank:pairwise needs group info — treat entire fold as one group
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval   = xgb.DMatrix(X_val, label=y_val)
        dtest  = xgb.DMatrix(X_test)

        dtrain.set_group([len(X_tr)])
        dval.set_group([len(X_val)])

        model = xgb.train(
            params,
            dtrain,
            num_boost_round      = params['n_estimators'],
            evals                = [(dval, 'val')],
            early_stopping_rounds= EARLY_STOPPING,
            verbose_eval         = False,
        )

        # Raw scores — rank-normalize to [0,1]
        val_scores  = model.predict(dval)
        test_scores = model.predict(dtest)

        oof_preds[val_idx] = val_scores
        test_preds        += test_scores / N_FOLDS

        fold_auc = roc_auc_score(y_val, val_scores)
        fold_aucs.append(fold_auc)
        print(f'  [RANK-s{seed}] Fold {fold}/{N_FOLDS} — AUC: {fold_auc:.5f}')

    oof_auc = roc_auc_score(y, oof_preds)
    print(f'  [RANK-s{seed}] OOF AUC: {oof_auc:.5f} | Mean: {np.mean(fold_aucs):.5f} ± {np.std(fold_aucs):.5f}')
    return oof_preds, test_preds, oof_auc

print('rank:pairwise CV function defined.')

rank:pairwise CV function defined.


In [ ]:
all_oof  = []
all_test = []
all_aucs = []

for seed in SEEDS:
    print(f'\nRunning rank:pairwise seed={seed}...')
    oof, test_preds, auc = run_cv_rank(seed)
    all_oof.append(oof)
    all_test.append(test_preds)
    all_aucs.append(auc)

#  Rank average across seeds 
oof_rank_avg  = np.mean([rankdata(o) / len(o) for o in all_oof],  axis=0)
test_rank_avg = np.mean([rankdata(t) / len(t) for t in all_test], axis=0)
auc_rank      = roc_auc_score(y, oof_rank_avg)

#  Prob average across seeds 
oof_prob_avg  = np.mean(all_oof,  axis=0)
test_prob_avg = np.mean(all_test, axis=0)
auc_prob      = roc_auc_score(y, oof_prob_avg)

print(f'\n  rank:pairwise Multi-seed Results:')
print(f'   Individual AUCs  : {[round(a, 5) for a in all_aucs]}')
print(f'   Prob avg OOF AUC : {auc_prob:.5f}')
print(f'   Rank avg OOF AUC : {auc_rank:.5f}')
print(f'\n   Previous best (binary:logistic rank avg): 0.97104')
print(f'   rank:pairwise improvement               : {auc_rank - 0.97104:+.5f}')


Running rank:pairwise seed=42...
  [RANK-s42] Fold 1/5 — AUC: 0.50000
  [RANK-s42] Fold 2/5 — AUC: 0.50000
  [RANK-s42] Fold 3/5 — AUC: 0.50000
  [RANK-s42] Fold 4/5 — AUC: 0.50000
  [RANK-s42] Fold 5/5 — AUC: 0.50000
  [RANK-s42] OOF AUC: 0.50003 | Mean: 0.50000 ± 0.00000

Running rank:pairwise seed=123...
  [RANK-s123] Fold 1/5 — AUC: 0.50000
  [RANK-s123] Fold 2/5 — AUC: 0.50000
  [RANK-s123] Fold 3/5 — AUC: 0.50000
  [RANK-s123] Fold 4/5 — AUC: 0.50000
  [RANK-s123] Fold 5/5 — AUC: 0.50000
  [RANK-s123] OOF AUC: 0.50003 | Mean: 0.50000 ± 0.00000

Running rank:pairwise seed=456...
  [RANK-s456] Fold 1/5 — AUC: 0.50000
  [RANK-s456] Fold 2/5 — AUC: 0.50000
  [RANK-s456] Fold 3/5 — AUC: 0.50000
  [RANK-s456] Fold 4/5 — AUC: 0.50000
  [RANK-s456] Fold 5/5 — AUC: 0.50000
  [RANK-s456] OOF AUC: 0.50003 | Mean: 0.50000 ± 0.00000

Running rank:pairwise seed=789...
  [RANK-s789] Fold 1/5 — AUC: 0.50000
  [RANK-s789] Fold 2/5 — AUC: 0.50000
  [RANK-s789] Fold 3/5 — AUC: 0.50000
  [RANK-s789

## 4. Blend rank:pairwise + binary:logistic

The two objectives learn different things — blending them can outperform either alone. We use the best binary:logistic predictions from Phase 7 and blend with rank:pairwise.

In [5]:
# Load Phase 7 best submission (XGB multi-seed rank avg, binary:logistic)
sub_v1 = pd.read_csv(os.path.join(OUT_DIR, 'submission_final.csv'))
logistic_test = sub_v1['order_placed'].values

# We need OOF from binary:logistic — rerun single seed to get OOF
# Use seed=42 binary:logistic params for OOF only
LOGISTIC_PARAMS = {
    'n_estimators'     : 1500,
    'learning_rate'    : 0.01381105528962201,
    'max_depth'        : 9,
    'min_child_weight' : 46,
    'subsample'        : 0.9280864631542471,
    'colsample_bytree' : 0.68577731284726,
    'reg_alpha'        : 2.75787843275108,
    'reg_lambda'       : 3.98468850175631,
    'scale_pos_weight' : 33.4,
    'eval_metric'      : 'auc',
    'use_label_encoder': False,
    'random_state'     : 42,
    'n_jobs'           : -1,
    'verbosity'        : 0,
}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_logistic = np.zeros(len(X))
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    model = xgb.XGBClassifier(**LOGISTIC_PARAMS)
    model.fit(
        X[tr_idx], y[tr_idx],
        eval_set              = [(X[val_idx], y[val_idx])],
        early_stopping_rounds = EARLY_STOPPING,
        verbose               = False
    )
    oof_logistic[val_idx] = model.predict_proba(X[val_idx])[:, 1]
    print(f'  [logistic-oof] Fold {fold}/{N_FOLDS} — AUC: {roc_auc_score(y[val_idx], oof_logistic[val_idx]):.5f}')

auc_logistic_oof = roc_auc_score(y, oof_logistic)
print(f'  Logistic OOF AUC: {auc_logistic_oof:.5f}')

# Rank-normalize both OOF predictions
oof_logistic_rank = rankdata(oof_logistic) / len(oof_logistic)
logistic_test_rank = rankdata(logistic_test) / len(logistic_test)

# Try blends
blend_results = {}
for w in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    blended_oof = w * oof_rank_avg + (1-w) * oof_logistic_rank
    blend_results[w] = roc_auc_score(y, blended_oof)

print('\nBlend results (rank:pairwise weight → OOF AUC):')
for w, auc in sorted(blend_results.items(), key=lambda x: -x[1]):
    marker = ' ← BEST' if auc == max(blend_results.values()) else ''
    print(f'  w={w:.1f} (rank:pairwise) + {1-w:.1f} (logistic): {auc:.5f}{marker}')

best_w    = max(blend_results, key=blend_results.get)
best_auc  = blend_results[best_w]
best_test = best_w * test_rank_avg + (1-best_w) * logistic_test_rank
print(f'\nBest blend weight: {best_w} rank:pairwise + {1-best_w} logistic')
print(f'Best blend AUC   : {best_auc:.5f}')

  [logistic-oof] Fold 1/5 — AUC: 0.96998
  [logistic-oof] Fold 2/5 — AUC: 0.97069
  [logistic-oof] Fold 3/5 — AUC: 0.96919
  [logistic-oof] Fold 4/5 — AUC: 0.97104
  [logistic-oof] Fold 5/5 — AUC: 0.97177
  Logistic OOF AUC: 0.97045

Blend results (rank:pairwise weight → OOF AUC):
  w=0.1 (rank:pairwise) + 0.9 (logistic): 0.96881 ← BEST
  w=0.2 (rank:pairwise) + 0.8 (logistic): 0.96375
  w=0.3 (rank:pairwise) + 0.7 (logistic): 0.95469
  w=0.4 (rank:pairwise) + 0.6 (logistic): 0.94057
  w=0.5 (rank:pairwise) + 0.5 (logistic): 0.91926
  w=0.6 (rank:pairwise) + 0.4 (logistic): 0.88588
  w=0.7 (rank:pairwise) + 0.3 (logistic): 0.82929
  w=0.8 (rank:pairwise) + 0.2 (logistic): 0.73570
  w=0.9 (rank:pairwise) + 0.1 (logistic): 0.61621

Best blend weight: 0.1 rank:pairwise + 0.9 logistic
Best blend AUC   : 0.96881


## 5. Save Best Submission

In [ ]:
# Determine what to submit
options = {
    'rank:pairwise rank avg'      : (auc_rank,     test_rank_avg),
    'rank:pairwise prob avg'      : (auc_prob,     test_prob_avg),
    f'blend (w={best_w})'         : (best_auc,     best_test),
}

print('Final comparison:')
print('-' * 55)
best_name, best_final_auc, best_final_preds = None, 0, None
for name, (auc, preds) in sorted(options.items(), key=lambda x: -x[1][0]):
    marker = ' ← BEST' if auc == max(o[0] for o in options.values()) else ''
    print(f'  {name:35s}: {auc:.5f}{marker}')
    if auc > best_final_auc:
        best_final_auc, best_name, best_final_preds = auc, name, preds

print(f'\n  Previous best (binary:logistic): 0.97104')
print(f'  rank:pairwise best             : {best_final_auc:.5f}')
print(f'  Improvement                    : {best_final_auc - 0.97104:+.5f}')

# Save primary
pd.DataFrame({
    'id'          : test['id'],
    'order_placed': best_final_preds
}).to_csv(SUB_PATH, index=False)
print(f'\n Saved: {SUB_PATH}')

# Also save rank:pairwise alone as backup
pd.DataFrame({
    'id'          : test['id'],
    'order_placed': test_rank_avg
}).to_csv(os.path.join(OUT_DIR, 'submission_rankpairwise_only.csv'), index=False)
print(f' Backup saved: submission_rankpairwise_only.csv')

Final comparison:
-------------------------------------------------------
  blend (w=0.1)                      : 0.96881 ← BEST
  rank:pairwise prob avg             : 0.49890
  rank:pairwise rank avg             : 0.49883

  Previous best (binary:logistic): 0.97104
  rank:pairwise best             : 0.96881
  Improvement                    : -0.00223

✅ Saved: c:\Users\User\OneDrive - Lebanese American University\University\SPRING 2026\COE 546 Machine Learning\Term Project\user_prediction_app\outputs\submission_rankpairwise.csv
✅ Backup saved: submission_rankpairwise_only.csv
